In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import math

## 多头注意力机制

$$Attention(Q,K,V) = softmax(\frac{QK^T}{\sqrt{d_k}})V$$
$$head_i = Attention(QW_i^Q, KW_i^K, VW_i^V)$$
$$MultiHeadAttention(Q,K,V) = Concat(head_1, head_2, ..., head_h)W^O$$

In [10]:
class ScaleDotProductAttention(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, q, k, v, mask=None):
        # shape: bsz * head * seq_len * head_size
        bsz, n_head, seq_len, hidden_size = q.size()

        k_t = k.transpose(2, 3)
        score = (q @ k_t) / math.sqrt(hidden_size)

        if mask:
            score = score.masked_fill(mask==0, -1e9)
        score = F.softmax(score, dim=-1)

        o = score @ v
        return o, score
        
class MultiHeadAttention(nn.Module):
    def __init__(self, hidden_size, n_head):
        super().__init__()
        self.n_head = n_head
        self.attention = ScaleDotProductAttention()

        self.w_q = nn.Linear(hidden_size, hidden_size)
        self.w_k = nn.Linear(hidden_size, hidden_size)
        self.w_v = nn.Linear(hidden_size, hidden_size)

        self.output = nn.Linear(hidden_size, hidden_size)

    def forward(self, q, k, v, mask=None):
        # bsz * sql_len * hidden_size
        q, k, v = self.w_q(q), self.w_k(k), self.w_v(v)

        # split: bsz * n_head * sql_len * head_size
        q, k, v = self.split(q), self.split(k), self.split(v)
        out, attention_score = self.attention(q, k, v, mask=mask)

        # concat: bsz * sql_len * hidden_size
        out = self.output(self.concat(out))
        return out

    def split(self, x):
        bsz, sql_len, hidden_size = x.size()

        head_size = hidden_size // self.n_head
        o = x.view(bsz, sql_len, self.n_head, head_size).transpose(1, 2)
        return o

    def concat(self, x):
        bsz, n_head, sql_len, head_size = x.size()
        o = x.transpose(1,2).contiguous().view(bsz, sql_len, n_head*head_size)
        return o
        

In [13]:
mha = MultiHeadAttention(512, 4)
x = torch.randn((2, 5, 512))
o = mha(x, x, x)

o.shape

torch.Size([2, 5, 512])

## FFN_SwiGLU

$$FFN(x) = f(xW_{up})W_{down}$$
$$FFN(x) = f(xW_{up} \odot xW_{gate})W_{down}$$
$f(*)$ - 激活函数，llama中使用F.silu

$$DOWN(UP(x) * ACT\_FN(GATE(x)))$$

In [19]:
class FeedForward(nn.Module):
    def __init__(self, hidden_size, intermediate_hidden_size, bias=0):
        super().__init__()
        self.gate = nn.Linear(hidden_size, intermediate_hidden_size, bias=bias)
        
        self.up   = nn.Linear(hidden_size, intermediate_hidden_size, bias=bias)
        self.down = nn.Linear(intermediate_hidden_size, hidden_size, bias=bias)
        self.act_fn = F.silu

    def forward(self, x):
        # x: bsz * sql_len * hidden_size
        o = self.down(self.act_fn(self.gate(x)) * self.up(x))
        return o

In [20]:
ffn = FeedForward(512, 512*4)
y = ffn(torch.randn((2, 5, 512)))
y.shape

torch.Size([2, 5, 512])

## EncoderLayer: attention -> Norm -> ffn -> Norm

$$LayerNorm(x + Norm(x))$$

In [25]:
# attention -> Norm -> ffn -> Norm

class EncoderLayer(nn.Module):
    def __init__(self, hidden_size, n_head, intermediate_hidden_size, drop_p=0):
        super().__init__()
        # attention part
        self.attention = MultiHeadAttention(hidden_size, n_head)
        self.dropout1  = nn.Dropout(p=drop_p)
        self.norm1 = nn.RMSNorm(hidden_size)

        # ffn part
        self.ffn = FeedForward(hidden_size, intermediate_hidden_size)
        self.dropout2 = nn.Dropout(p=drop_p)
        self.norm2 = nn.RMSNorm(hidden_size)

    def forward(self, x, mask=None):
        _x = x
        x = self.attention(q=x, k=x, v=x, mask=mask)
        x = self.norm1(self.dropout1(x) + _x)

        x = self.ffn(x)
        x = self.norm2(self.dropout2(x) + _x)
        return x

In [28]:
encoder = EncoderLayer(512, 4, 512 * 4)
y = encoder(torch.randn((2, 5, 512)))
y.shape

torch.Size([2, 5, 512])

## token embedding
$$PE_{(pos, 2i)} = sin(\frac{pos}{1000^{\frac{2i}{d_{model}}}})$$
$$PE_{(pos, 2i+1)} = cos(\frac{pos}{1000^{\frac{2i}{d_{model}}}})$$

In [70]:
class PositionEmbedding(nn.Module):
    def __init__(self, max_len, hidden_size):
        super().__init__()
        # import ipdb;ipdb.set_trace()
        self.embedding = torch.zeros(max_len, hidden_size)
        self.embedding.requires_grad = False

        pos = torch.arange(0, max_len)
        pos = pos.float().unsqueeze(dim=1)

        index = torch.arange(0, hidden_size, step=2).float()

        self.embedding[:, 0::2] = torch.sin(pos / (1000 ** index / hidden_size))
        self.embedding[:, 1::2] = torch.cos(pos / (1000 ** index / hidden_size))
        

    def forward(self, x):
        bsz, seq_len = x.size()
        return self.embedding[:seq_len, :]

class TransformerEmbedding(nn.Module):
    def __init__(self, vocab_size, hidden_size, max_len, drop_p):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, hidden_size)
        self.postion_embedding = PositionEmbedding(max_len, hidden_size)
        self.dropuout = nn.Dropout(p=drop_p)

    def forward(self, x):
        tok_emb = self.token_embedding(x)
        pos_emb = self.postion_embedding(x)

        return self.dropuout(tok_emb + pos_emb)

In [66]:
te = TransformerEmbedding(1000, 1024, 512, 0)
y = te(torch.randint(low=0, high=1000, size=(2,5), dtype=torch.int32))
y.shape

torch.Size([2, 5, 1024])

In [67]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, hidden_size, max_len, n_layers, n_head, intermediate_hidden_size, drop_p):
        super().__init__()
        self.emb = TransformerEmbedding(
            vocab_size=vocab_size,
            hidden_size=hidden_size,
            max_len=max_len,
            drop_p=drop_p,
        )
        self.layers = nn.ModuleList([
            EncoderLayer(
                hidden_size=hidden_size,
                n_head=n_head,
                intermediate_hidden_size=intermediate_hidden_size,
                drop_p=drop_p
            )
            for _ in range(n_layers)
        ])

    def forward(self, x, mask):
        x = self.emb(x)
        for layer in self.layers:
            x = layer(x, mask=mask)
        return x

In [68]:
encoder = Encoder(1000, 1024, 512, 5, 8, 1024*4, drop_p=0)

x = torch.randint(low=0, high=1000, size=(2,5), dtype=torch.int32)
x

tensor([[258, 664, 470, 635, 871],
        [657, 668, 195, 408, 856]], dtype=torch.int32)

In [69]:
o = encoder(x, mask=None)
o.shape, o

(torch.Size([2, 5, 1024]),
 tensor([[[-0.4014, -0.3572, -0.7034,  ...,  0.4324, -0.0154,  0.1133],
          [-1.4825, -0.6018,  0.0891,  ..., -0.2817,  0.0473,  0.2209],
          [ 0.2239,  0.4069, -1.5807,  ...,  0.3233, -0.5850,  0.7930],
          [ 0.2068, -0.4211, -0.9514,  ...,  2.1361,  1.4138,  1.1729],
          [-0.0714,  1.8944,  0.1436,  ...,  1.1942,  0.7152,  0.3771]],
 
         [[ 0.8254,  1.4739, -0.3327,  ..., -0.1294, -1.4219,  0.9532],
          [-0.8767,  1.5987,  0.5348,  ...,  1.5015, -1.1476,  0.0954],
          [ 0.0037, -0.3643,  0.1048,  ...,  1.5187, -0.1608,  1.8194],
          [-1.0904,  2.2241, -0.0616,  ...,  1.7537, -0.4426,  1.4895],
          [-0.2136,  0.1505, -0.0563,  ..., -0.6893,  2.2786,  1.0038]]],
        grad_fn=<MulBackward0>))

In [7]:
a = torch.randn((1, 8, 512))
b = torch.randn((8, 512, 128))

c = a @ b
c.size()

torch.Size([8, 8, 128])

In [ ]:
[2025-08-27 15:00:00] 	----------	Run Task: R2_1#779	----------
37231469610
78656844552